In [ ]:
#Here we repeat the Appendix C experiment with negative sampling

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rcParams
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML
import imageio_ffmpeg
mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
rcParams['animation.embed_limit'] = 100  # MB, default is 20
from sklearn.manifold import TSNE

from BadData_AppC import DataObject, AppendixCFunction, Trainer, Overlap
from NegSamplingMath_Unlearn_AppC_FullDataSet import NegMLP, NegTrainer, ActPoly, accuracy, NegOverlap

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def run_and_visualize_experiment(config: dict):
    """
    Runs a single experiment based on a configuration dictionary and visualizes the results.
    """
    print("="*80)
    print(f"Starting Experiment: {config['latex_title']} (p={config['p']})")
    print("="*80)

    # 1. Setup
    modular_function = AppendixCFunction(config['c'], config['d'], config['p'])
    dataset = DataObject(modular_function, split=config['split'])
    model = NegMLP(config['p'], config['embedding_dim'], config['hidden'])
    model = model.to(DEVICE)
    trainer = NegTrainer(learning_rate=config['learning_rate'], num_negs_per_example=config['negs_per_ex'])
    
    num_params = sum(p.numel() for p in model.parameters())
    space_dim = (config['p'] ** 2) * 2**4
    pos_dim = config['p'] ** 2
    print(f"num_parameters: {num_params:,}; space_dim: {space_dim:,}; pos_dim: {pos_dim:,}")
    
    if config.get('print_test_len', False):
        print(len(dataset.test_data))

    # 2. Training
    trainer.train_model(
        model,
        dataset,
        max_steps=config['max_steps'],
        batch_size=config['batch_size'],
        weight_decay=config['weight_decay']
    )

    # Polynomial as a string
    latex_title = config['latex_title']
    p = config['p']
    poly_name = config['poly_name']
    SaveTorF = config['SaveTorF']

    # 3. Visualization
    #PLot losses
    plt.figure()
    plt.plot(model.loss_dictionary['train_loss'], label = "Train Loss")
    plt.plot(model.loss_dictionary['test_loss'], label="Test Loss") 
    plt.xlabel("Training step")
    plt.ylabel("Loss")
    if len(dataset.test_data) == 1:
        plt.title('Loss ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
    else:
        plt.title('Loss ' + latex_title + ' mod ' + str(p), fontsize=10)
    plt.legend()
    if len(dataset.test_data) ==1 and SaveTorF == True:
        plt.savefig('Loss ' + poly_name + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), dpi=300, bbox_inches='tight')
    elif SaveTorF == True:
        plt.savefig('Loss ' + poly_name + ' mod ' + str(p), dpi=300, bbox_inches='tight')
    plt.show

    #Plot Accuracies
    plt.figure()
    plt.plot(model.loss_dictionary['train_accuracy'], label="Train Accuracy")
    plt.plot(model.loss_dictionary['test_accuracy'], label="Test Accuracy")
    plt.xlabel("Training step")
    plt.ylabel("Accuracy")
    if len(dataset.test_data) == 1:
        plt.title('Accuracy ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
    else:
        plt.title('Accuracy ' + latex_title + ' mod ' + str(p), fontsize=10)
    plt.legend()
    if len(dataset.test_data) ==1 and SaveTorF == True:
        plt.savefig('Accuracy ' + poly_name + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), dpi=300, bbox_inches='tight')
    elif SaveTorF == True:
        plt.savefig('Accuracy ' + poly_name + ' mod ' + str(p), dpi=300, bbox_inches='tight')   
    plt.show()

    #Plot Positive accuracies
    plt.figure()
    plt.plot(model.loss_dictionary['positive_train_accuracy'], label="Train Accuracy on positive examples")
    plt.plot(model.loss_dictionary['positive_test_accuracy'], label="Test Accuracy on positive examples")
    plt.xlabel("Training step")
    plt.ylabel("Accuracy")
    if len(dataset.test_data) == 1:
        plt.title('Accuracy on positive example ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
    else:
        plt.title('Accuracy on positive examples ' + latex_title + ' mod ' + str(p), fontsize=10)
    plt.legend()
    if len(dataset.test_data) ==1 and SaveTorF == True:
        plt.savefig('Accuracy on positive example ' + poly_name + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), dpi=300, bbox_inches='tight')
    elif SaveTorF == True:
        plt.savefig('Accuracy on positive examples ' + poly_name + ' mod ' + str(p), dpi=300, bbox_inches='tight')  
    plt.show()

    #Plot Negative accuracies
    plt.figure()
    plt.plot(model.loss_dictionary['negative_train_accuracy'], label="Train Accuracy on negative examples")
    plt.plot(model.loss_dictionary['negative_test_accuracy'], label="Test Accuracy on negative examples")
    plt.xlabel("Training step")
    plt.ylabel("Accuracy")
    if len(dataset.test_data) == 1:
        plt.title('Accuracy on negative example ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
    else:
        plt.title('Accuracy on negative examples ' + latex_title + ' mod ' + str(p), fontsize=10)
    plt.legend()
    if len(dataset.test_data) ==1 and SaveTorF == True:
        plt.savefig('Accuracy on negative examples ' + poly_name + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), dpi=300, bbox_inches='tight')
    elif SaveTorF == True:
        plt.savefig('Accuracy on negative examples ' + poly_name + ' mod ' + str(p), dpi=300, bbox_inches='tight')  
    plt.show()

    #Evolution of t-SNE projection of word and context vectors
    #First word vectors
    #Evolution of t-SNE of the word embedding vectors
    embs = []
    for E in model.loss_dictionary.get('embedding_matrix', []):
        arr = E.detach().cpu().numpy() if isinstance(E, torch.Tensor) else np.array(E)
        embs.append(arr)

    if embs:
        # Stack all snapshots for one big TSNE so the 2D layout stays consistent
        all_E = np.concatenate(embs, axis=0)   # shape = (steps * p, embedding_dim)

        # Run TSNE once
        tsne = TSNE(n_components=2, init="pca", perplexity=min(30, p-1))
        all_2d = tsne.fit_transform(all_E)

        # Reshape back into a list of frames: (num_steps, p, 2)
        steps = len(embs)
        p_embed = embs[0].shape[0]
        frames_2d = all_2d.reshape(steps, p_embed, 2)

        # Set up animation
        fig, ax = plt.subplots(figsize=(6,6))
        ax.set_title(f"t-SNE word embedding evolution {latex_title} mod {p}")
        scat = ax.scatter(frames_2d[0][:,0], frames_2d[0][:,1], s=10)

        # Fix axes so they don't jump between frames
        ax.set_xlim(all_2d[:,0].min(), all_2d[:,0].max())
        ax.set_ylim(all_2d[:,1].min(), all_2d[:,1].max())

        def update(i):
            scat.set_offsets(frames_2d[i])
            ax.set_title(f"t-SNE word embedding evolution {latex_title} mod {p} (step {i+1}/{steps})")
            return scat,

        anim = FuncAnimation(fig, update, frames=steps, interval=120, repeat=False)

        if SaveTorF:
            anim.save('tsne_evolution_word_embeddings_' + poly_name + '_mod_' + str(p) + '.mp4',
                    writer="ffmpeg", dpi=150, fps=25)

        plt.close(fig)
        display(HTML(anim.to_jshtml()))
    else:
        print("No embeddings found in model.loss_dictionary['embedding_matrix'].")


    # Now context vectors
    # Evolution of t-SNE of the context embedding vectors (per-step TSNE)
    context_snapshots = model.loss_dictionary.get('context_vecs', [])

    if context_snapshots:
        tsne_frames = []

        for C in context_snapshots:
            # C is a tensor of context vectors for ONE step
            arr = C.detach().cpu().numpy() if isinstance(C, torch.Tensor) else np.array(C)

            # If shape is (N, 1, d), flatten to (N, d)
            arr = arr.reshape(arr.shape[0], -1)

            # Optional: downsample rows if you really have p^2 and it's huge
            max_points = 5000  # change or remove if you want all
            if arr.shape[0] > max_points:
                idx = np.random.choice(arr.shape[0], size=max_points, replace=False)
                arr = arr[idx]

            tsne = TSNE(
                n_components=2,
                init="pca",
                perplexity=min(30, arr.shape[0] - 1),
                learning_rate="auto",
                random_state=42,
            )
            coords_2d = tsne.fit_transform(arr)  # shape (n_points, 2) for THIS step
            tsne_frames.append(coords_2d)

        # Use global limits so the view doesn't jump
        all_2d = np.concatenate(tsne_frames, axis=0)
        x_min, x_max = all_2d[:,0].min(), all_2d[:,0].max()
        y_min, y_max = all_2d[:,1].min(), all_2d[:,1].max()

        # Set up animation
        fig, ax = plt.subplots(figsize=(8, 6))
        scat = ax.scatter(tsne_frames[0][:,0], tsne_frames[0][:,1], s=10)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")

        def update(i):
            scat.set_offsets(tsne_frames[i])
            ax.set_title(
                f"t-SNE context embedding evolution {latex_title} mod {p} "
                f"(frame {i+1}/{len(tsne_frames)})",
                fontsize=10,
            )
            return scat,

        anim = FuncAnimation(fig, update, frames=len(tsne_frames), interval=120, repeat=False)

        if SaveTorF:
            anim.save(
                'tsne_evolution_context_embeddings_' + poly_name + '_mod_' + str(p) + '.mp4',
                writer="ffmpeg", dpi=150, fps=25
            )

        plt.close(fig)
        display(HTML(anim.to_jshtml()))
    else:
        print("No context_vecs found in model.loss_dictionary; skipping t-SNE.")


    # Animation for single-point test sets
    if len(dataset.test_data) == 1:

        #rcParams['animation.embed_limit'] = 64  # MB, default is 20

        # Histogram
        example = dataset.test_data
        print(example)
        plt.figure()
        plt.bar(list(range(p)), model.loss_dictionary['counts_hist_all'])
        plt.title("Histogram " + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
        plt.xlabel("Question index")
        plt.ylabel("Count of predictions")
        plt.draw()
        plt.pause(0.001)
        if SaveTorF == True:
            plt.savefig("Histogram " + poly_name + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), dpi=300, bbox_inches='tight')
        plt.show

        #Histogram with single prediction
        plt.figure()
        plt.bar(list(range(p)), model.loss_dictionary['prediction_mike'])
        plt.title("Histogram for single prediction " + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)
        plt.xlabel("Question index")
        plt.ylabel("Count of predictions")
        plt.draw()
        plt.pause(0.001)
        if SaveTorF == True:
            plt.savefig("Histogram for single prediction " + poly_name + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), dpi=300, bbox_inches='tight')
        plt.show

        # Evolution of probability distribution
        probs = []
        for f in model.loss_dictionary.get('sorta_prob_dist', []):
            arr = f.detach().cpu().squeeze().numpy() if isinstance(f, torch.Tensor) else np.array(f).squeeze()
            s = float(arr.sum())
            if s > 0:  
                arr = arr / s
            else:
                print("Warning: Sum of probabilities is zero.")
            probs.append(arr)

        if probs:
            fig, ax = plt.subplots(figsize=(6,3))
            ax.set_title('Probability distribution evolution ' + latex_title + ' mod ' + str(p) + ' for ' + str(dataset.test_data.tolist()), fontsize=10)   
            bars = ax.bar(range(len(probs[0])), probs[0])
            ax.set_ylim(0, 0.025) 
            txt = ax.text(0.02, 0.95, '', transform=ax.transAxes)

            def update(i):
                y = probs[i]
                for b, h in zip(bars, y): 
                    b.set_height(float(h))
                txt.set_text(f"step {i+1}/{len(probs)} | sum={y.sum():.3f} | argmax={int(np.argmax(y))}")
                return bars
            
            anim = FuncAnimation(fig, update, frames=len(probs), interval=100, repeat=False)
            if SaveTorF == True:
                anim.save('probability_evolution_' + poly_name + '_mod_' + str(p) + '.mp4', writer="ffmpeg", dpi=150, fps=25)
            plt.close(fig)
            display(HTML(anim.to_jshtml()))
    
    print("\n✅ Experiment Complete.\n")


In [ ]:
experiment_mod_add = [
    {
    # Polynomial x + y % p
        "poly_name": "Modular Addition", "p": 31, "c": [1, 1, 0], "d": [1, 1, 1, 0, 0], "latex_title": r"$x + y$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.5, "negs_per_ex": 10 , 'SaveTorF': False
    }]

default_training_params = {
    "max_steps": 1000,
    "learning_rate": 0.005,
    "batch_size": 1024,
    "weight_decay": 1e-4,
}

In [ ]:
for exp_config in experiment_mod_add:
    final_config = {**default_training_params, **exp_config} # Merge dictionaries
    run_and_visualize_experiment(final_config)

In [ ]:
#Polynomials modulo 53

experiments = [
    {
    # Polynomial (4*x + y**2)**3 % 53
        "poly_name": "Polynomial 1", "p": 53, "c": [4, 1, 0], "d": [1, 2, 3, 0, 0], "latex_title": r"$(4x + y^2)^3$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.5, "negs_per_ex": 10 , 'SaveTorF': False
    },
    # Polynomial (4*x + y**2)**3 + xy % 53
    {
        "poly_name": "Polynomial 2", "p": 53, "c": [4, 1, 1], "d": [1, 2, 3, 1, 1], "latex_title": r"$(4x + y^2)^3 + xy$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.5, "negs_per_ex": 10 , 'SaveTorF': False
    },
    # Polynomial (2*x + 3*y)**4 % 53
    {
        "poly_name": "Polynomial 3", "p": 53, "c": [2, 3, 0], "d": [1, 1, 4, 0, 0], "latex_title": r"$(2x + 3y)^4$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.5, "negs_per_ex": 10 , 'SaveTorF': False
    },
    # Polynomial (2*x + 3*y)**4 - x**2 % 53
    {
        "poly_name": "Polynomial 4", "p": 53, "c": [2, 3, -1], "d": [1, 1, 4, 2, 0], "latex_title": r"$(2x + 3y)^4 - x^2$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.5, "negs_per_ex": 10 , 'SaveTorF': False
    },
    # Polynomial (x**3 + 2*y**4)**2 % 53
    { 
        "poly_name": "Polynomial 5", "p": 53, "c": [5, 2, 0], "d": [3, 4, 2, 0, 0], "latex_title": r"$(5x^3 + 2y^4)^2$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.5, "negs_per_ex": 10 , 'SaveTorF': False
    },
    # Polynomial (x**3 + 2*y**4)**2 - y % 53
    {
        "poly_name": "Polynomial 6", "p": 53, "c": [5, 2, -1], "d": [3, 4, 2, 0, 1], "latex_title": r"$(5x^3 + 2y^4)^2 - y$",
        "embedding_dim": 2**7, "hidden": 500, "split": 0.5, "negs_per_ex": 10 , 'SaveTorF': False
    }
    # test
]

default_training_params = {
    "max_steps": 1000,
    "learning_rate": 0.005,
    "batch_size": 1024,
    "weight_decay": 1e-4,
}


In [ ]:
for exp_config in experiments:
    final_config = {**default_training_params, **exp_config} # Merge dictionaries
    run_and_visualize_experiment(final_config)